# Shapflow with Data-Driven Graph Construction

Adapting shapflow.flow to work with discovered causal structures by using **empirical conditional distributions** instead of theoretical functions.

**Key Insight from Tutorial:**
- Source nodes (no parents): Just names, no functions needed
- Intermediate nodes (with parents): Need functions - we'll use data-driven sampling
- Target node (Y): Use the ML model

This bridges the gap between the tutorial (known mechanisms) and our case (discovered structure only).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import sys

# Add shapflow to path if needed
shapflow_path = Path.cwd() / 'shapflow'  # Adjust if shapflow is elsewhere
if shapflow_path.exists() and str(shapflow_path) not in sys.path:
    sys.path.insert(0, str(shapflow_path))

from shapflow.flow import Node, Graph, GraphExplainer
from predictive_models.predictive_models import LGBMRegressor

print("✅ Imports successful")

✅ Imports successful


## 1. Load Data and Model

In [2]:
# Dataset configuration
dataset_name = "mixed_no_conf_f50_s1000_p50"

# Load data
train_data = pd.read_parquet(f"data/processed/{dataset_name}_train.parquet")
test_data = pd.read_parquet(f"data/processed/{dataset_name}_test.parquet")

X_train = train_data.drop('Y', axis=1)
y_train = train_data['Y']
X_test = test_data.drop('Y', axis=1)
y_test = test_data['Y']

# Load model
model = LGBMRegressor.load(f"models/{dataset_name}_lgbm")

print(f"Data: {X_train.shape}, Model features: {len(model.selected_features)}")

LightGBM model loaded from models/mixed_no_conf_f50_s1000_p50_lgbm.pkl
Data: (800, 50), Model features: 50


In [3]:
train_data["Y"].describe()

count    800.000000
mean       0.014874
std        1.100613
min       -7.638759
25%       -0.475177
50%       -0.034988
75%        0.417622
max        7.271624
Name: Y, dtype: float64

In [4]:
# Load causal structure
# adjacency_matrix = np.load(f"data/causal/{dataset_name}_pc_train_adjacency.npy")
import json
results_path = f"data/causal/{dataset_name}_pc_results.json"
with open(results_path, 'r') as f:
    results_json = json.load(f)
adjacency_matrix = np.array(results_json['adjacency_matrix'])
n_features = adjacency_matrix.shape[0]

print(f"Causal structure: {adjacency_matrix.shape}")
print(f"Edges: {np.sum(adjacency_matrix != 0)}")

Causal structure: (51, 51)
Edges: 80


## 2. Build Data-Driven Conditional Samplers

Since we don't know the true causal functions, we'll create empirical samplers that learn from data.

In [5]:
def adjacency_to_graph(adjacency_matrix, feature_names, train_data):
    """
    Convert adjacency matrix to shapflow Graph object.
    
    Parameters:
    -----------
    adjacency_matrix : np.ndarray
        Adjacency matrix where adjacency[i, j] != 0 means feature i is a parent of feature j
    feature_names : list
        List of feature names (e.g., ['X1', 'X2', ..., 'Y'])
    train_data : pd.DataFrame
        Training data to fit the causal functions
    
    Returns:
    --------
    Graph : shapflow.flow.Graph object with learned causal functions
    """
    n_features = len(feature_names)
    
    # Step 1: Create all nodes first (without parent relationships)
    nodes_dict = {}
    for i, name in enumerate(feature_names):
        # Determine if this is the target node
        is_target = (name == 'Y')
        
        # Create node (args will be added later)
        node = Node(
            name=name,
            f=None,
            args=[],
            is_target_node=is_target,
            is_noise_node=False,
            is_dummy_node=False,
            is_categorical=False
        )
        nodes_dict[name] = node
    
    # Step 2: Add parent relationships based on adjacency matrix
    for j, child_name in enumerate(feature_names):
        child_node = nodes_dict[child_name]
        
        # Find parents: where adjacency[i, j] != 0 means i is parent of j
        parent_indices = np.where(adjacency_matrix[:, j] != 0)[0]
        
        # Add each parent to the child's args
        for parent_idx in parent_indices:
            parent_name = feature_names[parent_idx]
            parent_node = nodes_dict[parent_name]
            child_node.args.append(parent_node)
            # Also add child to parent's children list
            if child_node not in parent_node.children:
                parent_node.children.append(child_node)
    
    # Step 3: Create Graph object
    graph = Graph(nodes=list(nodes_dict.values()))
    
    # Step 4: Fit missing links (learn causal functions from data)
    print("Learning causal functions from training data...")
    graph.fit_missing_links(train_data, method='xgboost')
    
    return graph

print("✅ Helper function created")

✅ Helper function created


## Step 1: Create Graph from Adjacency Matrix

In [6]:
# Get feature names from train_data
feature_names = list(train_data.columns)
print(f"Features: {len(feature_names)} total")
print(f"Feature names: {feature_names[:5]}... Y")

# Create the Graph from adjacency matrix
# This will create nodes and learn causal functions
graph = adjacency_to_graph(adjacency_matrix, feature_names, train_data)

print(f"\n✅ Graph created with {len(graph.nodes)} nodes")
print(f"Source nodes (no parents): {sum(1 for n in graph.nodes if len(n.args) == 0)}")
print(f"Target node: {[n.name for n in graph.nodes if n.is_target_node]}")

Features: 51 total
Feature names: ['X0', 'X1', 'X2', 'X3', 'X4']... Y
Learning causal functions from training data...


learning dependency for X0:   0%|          | 0/49 [00:00<?, ?it/s]

[0]	test-rmse:0.52133
[100]	test-rmse:0.45091
[200]	test-rmse:0.39575
[300]	test-rmse:0.35338
[400]	test-rmse:0.32090
[499]	test-rmse:0.29723


learning dependency for X5:   2%|▏         | 1/49 [00:00<00:27,  1.74it/s]

[0]	test-rmse:1.05327
[100]	test-rmse:0.96581
[200]	test-rmse:0.89755
[300]	test-rmse:0.84495
[400]	test-rmse:0.80397
[499]	test-rmse:0.77342


...

[0]	test-rmse:0.99652
[100]	test-rmse:0.89082
[200]	test-rmse:0.80596
[300]	test-rmse:0.73929
[400]	test-rmse:0.68703
[499]	test-rmse:0.64842


learning dependency for X22:  98%|█████████▊| 48/49 [00:27<00:00,  1.77it/s]

[0]	test-rmse:0.95009
[100]	test-rmse:0.93433
[200]	test-rmse:0.92309
[300]	test-rmse:0.91710
[400]	test-rmse:0.91388
[499]	test-rmse:0.91187


learning dependency for X22: 100%|██████████| 49/49 [00:27<00:00,  1.77it/s]


✅ Graph created with 51 nodes
Source nodes (no parents): 2
Target node: ['Y']


In [7]:
[node for node in graph.nodes if node.is_target_node]

[Y]

## Step 3: Create GraphExplainer

Now we'll create the GraphExplainer with:
- Background: 100 random samples from training data
- Foreground: First 2 test instances

In [ ]:
# Prepare background data: 100 random samples from training data (including Y)
np.random.seed(42)
bg_indices = np.random.choice(len(train_data), size=100, replace=False)
background_data = train_data.iloc[bg_indices]

# Prepare foreground data: First 2 test instances (including Y)
TEST_INSTANCES_RATIO = 0.5 # Use 50% of test data for foreground
RANDOM_STATE = 42   
foreground_data = test_data.sample(n=int(len(test_data) * TEST_INSTANCES_RATIO), random_state=RANDOM_STATE)

print(f"Background data: {background_data.shape}")
print(f"Foreground data: {foreground_data.shape}")
print(f"\nForeground instances:")
print(foreground_data[['Y']].head())

# Create GraphExplainer
explainer = GraphExplainer(
    graph=graph,
    bg=background_data,
    nruns=100,  # Number of Monte Carlo samples
    silent=False
)

print(f"\n✅ GraphExplainer created successfully!")

Background data: (100, 51)
Foreground data: (200, 51)

Foreground instances:
            Y
521 -0.032315
737 -0.972250
740  0.343082
660  0.444555
411 -0.157881

✅ GraphExplainer created successfully!


In [60]:
test_data.shape

(200, 51)

## Test: Compute SHAP Values

Let's compute the Shapley Flow values for our test instances:

In [20]:
# Compute SHAP values using the GraphExplainer
# This uses the Shapley Flow algorithm with the learned causal graph
cf = explainer.shap_values(
    X=foreground_data,
    method='bruteforce_sampling',  # Monte Carlo sampling method
    # method_type='distributed'  # Use distributed edge axioms
)

print("\n✅ Shapley Flow computation complete!")
print(f"Edge credits computed: {len(cf.edge_credit)} parent nodes")

bruteforce sampling: 100%|██████████| 100/100 [01:25<00:00,  1.16it/s]


✅ Shapley Flow computation complete!
Edge credits computed: 33 parent nodes


In [37]:
def get_node_attributions(edge_credit):
        """
        Aggregate edge attributions to get node-level importance

        Returns: 
        --------
        node_attr :
            Node importance scores (sum of outgoing edge attributions)
        """

        node_attr = {}
        for node1, d in edge_credit.items():
            if "noise" not in node1.name:
                for node2, val in d.items():
                    node_attr[node1.name] = node_attr.get(node1, 0.0) + val
        return node_attr

In [ ]:
def convert_node_attributions_to_array(node_attributions, feature_names):
    """
    Convert node attributions dictionary to standard SHAP format array.
    
    Parameters:
    -----------
    node_attributions : dict
        Dictionary from get_node_attributions with format {feature_name: array_of_shap_values}
        where each array has shape (n_instances,)
    feature_names : list
        List of all feature names in the correct order (e.g., from X.columns)
        Should NOT include 'Y' (target)
    
    Returns:
    --------
    shap_array : np.ndarray
        Array of shape (n_instances, n_features) with SHAP values
        Features not in node_attributions get value 0
    """
    # Remove 'Y' from feature names if present
    feature_names = [f for f in feature_names if f != 'Y']
    
    # Get number of instances from any feature in the dictionary
    if len(node_attributions) > 0:
        first_feature = list(node_attributions.keys())[0]
        n_instances = len(node_attributions[first_feature])
    else:
        raise ValueError("node_attributions is empty")
    
    n_features = len(feature_names)
    
    # Initialize array with zeros
    shap_array = np.zeros((n_instances, n_features))
    
    # Fill in SHAP values for features that have them
    for i, feature_name in enumerate(feature_names):
        if feature_name in node_attributions:
            shap_array[:, i] = node_attributions[feature_name]
        # else: remains 0
    
    return shap_array

print("✅ Conversion helper function created")

### Convert to Standard SHAP Array Format (100 × 50)

In [ ]:
# Get node attributions (17 features x 100 instances dictionary)
shap_values_dict = get_node_attributions(cf.edge_credit)

print(f"Dictionary format: {len(shap_values_dict)} features")
print(f"Sample feature shapes:")
for i, (feature, values) in enumerate(list(shap_values_dict.items())[:3]):
    print(f"  {feature}: {values.shape}")

# Convert to standard array format (100 instances x 50 features)
X_feature_names = [col for col in feature_names if col != 'Y']
shap_values_array = convert_node_attributions_to_array(shap_values_dict, X_feature_names)

print(f"\n✅ Converted to array format: {shap_values_array.shape}")
print(f"   Shape: (n_instances={shap_values_array.shape[0]}, n_features={shap_values_array.shape[1]})")
print(f"\nFeatures with non-zero SHAP values: {np.sum(np.any(shap_values_array != 0, axis=0))}")
print(f"Features with zero SHAP values: {np.sum(np.all(shap_values_array == 0, axis=0))}")

In [ ]:
# Verify the conversion: Check that feature order matches
print("Feature Order Verification:")
print("=" * 70)
print(f"Total features in X: {len(X_feature_names)}")
print(f"First 5 features: {X_feature_names[:5]}")
print(f"Last 5 features: {X_feature_names[-5:]}")

# Check a specific feature's values match
test_feature = list(shap_values_dict.keys())[0]  # Take first feature from dict
feature_idx = X_feature_names.index(test_feature)

print(f"\nVerification for feature '{test_feature}':")
print(f"  Position in array: column {feature_idx}")
print(f"  Dict values (first 5 instances): {shap_values_dict[test_feature][:5]}")
print(f"  Array values (first 5 instances): {shap_values_array[:5, feature_idx]}")
print(f"  Match: {np.allclose(shap_values_dict[test_feature], shap_values_array[:, feature_idx])}")

# Check that features not in DAG have zero values
non_dag_features = [f for f in X_feature_names if f not in shap_values_dict]
print(f"\nFeatures NOT in DAG (should be all zeros): {len(non_dag_features)}")
if non_dag_features:
    print(f"  Examples: {non_dag_features[:5]}")
    # Verify they're zeros
    test_non_dag = non_dag_features[0]
    test_idx = X_feature_names.index(test_non_dag)
    print(f"  '{test_non_dag}' values: {shap_values_array[:5, test_idx]}")
    print(f"  All zeros: {np.all(shap_values_array[:, test_idx] == 0)}")

In [ ]:
# Now you can use it like standard SHAP values!
# Example: Get mean absolute SHAP values for feature importance
mean_abs_shap = np.mean(np.abs(shap_values_array), axis=0)

# Create a DataFrame for easier analysis
feature_importance_df = pd.DataFrame({
    'Feature': X_feature_names,
    'Mean_Abs_SHAP': mean_abs_shap
}).sort_values('Mean_Abs_SHAP', ascending=False)

print("\nTop 15 Most Important Features (by mean absolute SHAP):")
print("=" * 70)
print(feature_importance_df.head(15).to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
top_20 = feature_importance_df.head(20)
ax.barh(range(len(top_20)), top_20['Mean_Abs_SHAP'])
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20['Feature'])
ax.set_xlabel('Mean Absolute SHAP Value')
ax.set_title('Top 20 Feature Importance (Shapley Flow)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\n✅ Standard format ready! Shape: {shap_values_array.shape}")

In [ ]:
# Example usage: Access SHAP values like other methods
print("Usage Examples:")
print("=" * 70)

# Get SHAP values for a specific instance (e.g., instance 0)
instance_idx = 0
instance_shap = shap_values_array[instance_idx, :]
print(f"Instance {instance_idx} SHAP values shape: {instance_shap.shape}")
print(f"Top 5 contributors for instance {instance_idx}:")
top_features_inst = np.argsort(np.abs(instance_shap))[-5:][::-1]
for feat_idx in top_features_inst:
    print(f"  {X_feature_names[feat_idx]:10s}: {instance_shap[feat_idx]:10.4f}")

# Get SHAP values for a specific feature (e.g., first DAG feature)
feature_name = list(shap_values_dict.keys())[0]
feature_idx = X_feature_names.index(feature_name)
feature_shap = shap_values_array[:, feature_idx]
print(f"\nFeature '{feature_name}' SHAP values across all instances:")
print(f"  Shape: {feature_shap.shape}")
print(f"  Mean: {feature_shap.mean():.4f}")
print(f"  Std: {feature_shap.std():.4f}")
print(f"  Range: [{feature_shap.min():.4f}, {feature_shap.max():.4f}]")

### Summary: SHAP Values Format

**Original Dictionary Format (from `get_node_attributions`):**
- Structure: `{feature_name: array_of_100_shap_values}`
- Only 17 features (those in the DAG)
- Example: `{'X1': [shap_inst0, shap_inst1, ...], 'X5': [...], ...}`

**Converted Array Format (standard SHAP format):**
- Structure: `np.ndarray` of shape `(100, 50)`
- **Rows**: 100 instances (each row = one instance)
- **Columns**: 50 features (in same order as original X data)
- Features NOT in DAG have SHAP value = 0
- **Compatible** with all standard SHAP visualization and analysis tools!

**Usage:**
```python
# Get SHAP values for instance i, feature j
shap_value = shap_values_array[i, j]

# Get all SHAP values for instance i
instance_values = shap_values_array[i, :]

# Get all SHAP values for feature j across all instances
feature_values = shap_values_array[:, j]
```

In [55]:
shap_values2=get_node_attributions(cf.edge_credit)
shap_values2

{'X30': array([ 1.40334386e-03, -2.45743986e-03, -1.31512024e-02,  5.43912237e-03,
         9.92700290e-04,  3.26192652e-03, -1.15014363e-01, -1.16175711e-01,
        -1.31512024e-02,  7.72013367e-03, -3.13427458e-02,  7.17719737e-03,
         2.24915176e-02,  0.00000000e+00,  0.00000000e+00,  2.38829264e-02,
        -9.34183237e-02,  3.92191601e-03, -7.17539482e-03, -4.28732513e-02,
        -3.50600732e-02, -4.15933233e-02,  2.24915185e-02,  0.00000000e+00,
         1.89363491e-03, -9.17928619e-02, -2.71923872e-02, -9.99787611e-02,
        -1.31512024e-02,  5.43912188e-03,  6.51404727e-03,  1.89363551e-03,
        -2.63145572e+00,  0.00000000e+00, -1.05254726e-02,  0.00000000e+00,
         9.92699261e-04, -1.31512024e-02, -1.01334695e-01,  6.69810471e-03,
        -4.54365792e-02, -2.45744095e-03,  9.92699200e-04, -1.31512022e-02,
        -2.77117113e-02,  9.92698595e-04,  0.00000000e+00, -2.45744095e-03,
         9.92698595e-04,  4.93163476e-02,  0.00000000e+00, -5.53631512e-02,
     

In [ ]:
{k: v.mean() for k, v in get_node_attributions(cf.edge_credit).items( )}

{'X30': np.float64(-0.03214336529207066),
 'X48': np.float64(-0.00499163632488926),
 'X18': np.float64(-0.003100617777358275),
 'X16': np.float64(-0.001422732543328311),
 'X21': np.float64(8.414646203164011e-07),
 'X33': np.float64(1.5110059175640344e-07),
 'X5': np.float64(-3.2218519365414973e-06),
 'X14': np.float64(-0.0035633540896880733),
 'X22': np.float64(-2.0428726961836223e-05),
 'X17': np.float64(-4.2102180363144726e-06),
 'X4': np.float64(-2.4763529771007593e-06),
 'X32': np.float64(-0.006811807765586126),
 'X24': np.float64(0.0020916838757148067),
 'X23': np.float64(3.3502423320896923e-06),
 'X2': np.float64(-3.4039840102195738e-09),
 'X8': np.float64(-3.8148080930113797e-07),
 'X15': np.float64(-1.3822905020788312e-07)}

In [44]:
# # cf.print_credit()
# edge_credit = cf.edge_credit
# for node1, d in edge_credit.items():
#     if "noise" not in node1.name:
#         for node2, val in d.items():
#             print(f'credit {node1}->{node2}: {val}')

## Extract Feature-Level SHAP Values

The `get_asv_edge_credit()` method aggregates edge credits to feature-level attributions (SHAP values):

In [52]:
# Get feature-level attributions (aggregates edge credits to source features only)
# idx=-1 means aggregate over all instances (if you had multiple)
# idx=0 would give you attributions for the first instance
feature_credits = cf.get_asv_edge_credit(idx=-1,aggregate=False)

print("Feature-Level SHAP Values:")
print("=" * 60)

# Extract feature → target attributions
shap_values = {}
target_node = [node for node in cf.graph.nodes if node.is_target_node][0]

for source_node, credits in feature_credits.items():
    if "noise" in source_node.name:
        continue
    if target_node in credits:
        feature_name = source_node.name
        attribution = credits[target_node]
        shap_values[feature_name] = attribution
        # print(f"{feature_name:20s}: {attribution:10.6f}")

print("=" * 60)
print(f"Total features with attributions: {len(shap_values)}")

Feature-Level SHAP Values:
Total features with attributions: 2


In [57]:
shap_values2["X21"]

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.00016829, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.     

In [58]:
# Convert to pandas DataFrame for easier analysis
shap_df = pd.DataFrame.from_dict(shap_values, orient='index', columns=['SHAP_Value'])
shap_df.index.name = 'Feature'
shap_df = shap_df.sort_values('SHAP_Value', ascending=False, key=abs)

print("\nTop 15 Most Important Features (by absolute SHAP value):")
print(shap_df.head(15))

# Visualize
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 8))
top_features = shap_df.head(20)
colors = ['red' if x > 0 else 'blue' for x in top_features['SHAP_Value']]
ax.barh(range(len(top_features)), top_features['SHAP_Value'], color=colors)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features.index)
ax.set_xlabel('SHAP Value (Feature Attribution)')
ax.set_title('Top 20 Feature Attributions from Shapley Flow')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

print(f"\n✅ Feature-level SHAP values extracted successfully!")

ValueError: 1 columns passed, passed data had 200 columns

### Understanding Edge Credits vs Feature Attributions

**Edge Credits** (`cf.edge_credit`):
- Low-level: Shows credit flow between ALL nodes in the causal graph
- Includes intermediate nodes (features with parents)
- Format: `{parent_node: {child_node: credit_value}}`

**Feature Attributions** (`cf.get_asv_edge_credit()`):
- High-level: Aggregates to show only **source features → target**
- "ASV" = Additive Shapley Values (source features only)
- Format: `{source_feature: {target: SHAP_value}}`
- This is what you typically want for feature importance!

In [59]:
# Optional: Get per-instance SHAP values (if you have multiple instances)
# For instance 0:
instance_idx = 0
feature_credits_inst0 = cf.get_asv_edge_credit(idx=instance_idx, aggregate=False)

print(f"SHAP values for instance {instance_idx}:")
print("=" * 60)

shap_values_inst0 = {}
for source_node, credits in feature_credits_inst0.items():
    if target_node in credits:
        feature_name = source_node.name
        attribution = credits[target_node]
        shap_values_inst0[feature_name] = attribution
        
# Sort by absolute value
sorted_features = sorted(shap_values_inst0.items(), key=lambda x: abs(x[1]), reverse=True)
for feature, value in sorted_features[:10]:
    print(f"{feature:20s}: {value:10.6f}")

print("=" * 60)

SHAP values for instance 0:
Y noise             :   0.423324
X30 noise           :   0.001403
X21                 :   0.000000
X5 noise            :   0.000000
X33 noise           :   0.000000
X23 noise           :   0.000000
X4 noise            :   0.000000
X24 noise           :   0.000000
X48 noise           :   0.000000
X22 noise           :   0.000000


In [15]:
pip install pygraphviz

/Users/juanrios/Documents/master_thesis/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
